# No-show Prediction

This notebook analyzes healthcare appointment data to predict patient no-shows. The workflow includes:

- Reading and loading the dataset
- Data cleaning and schema definition
- Handling missing values and dropping unnecessary columns
- Identifying and scaling numeric features for modeling

The goal is to prepare the data for building predictive models to identify factors influencing patient attendance.

### Setup cells commented out for use in Jobs pipeline

In [0]:
%skip
%run ./setup/import_dataset

In [0]:
%skip
%run ./setup/data_preprocessing

Read in `default.silver_no_show_features`

In [0]:
silver_df = spark.table("workspace.default.silver_no_show_features")

#### Train-test split

In [0]:
train_df, test_df = silver_df.randomSplit([0.8, 0.2], seed=42)
print(f"Train: {train_df.count():,}  Test: {test_df.count():,}")


### Vector Assembler

In [0]:
# Base numeric features (cast to double in the cell above, IDs and target excluded)
numerical_cols = ["Age", "Scholarship", "Hypertension", "Diabetes",
                  "Alcoholism", "Handicap", "SMS_received", "date_diff"]

### Cyclical dates

In [0]:
import math
from pyspark.sql.functions import col, sin, cos, month, dayofweek, dayofyear, lit

for df_name, df in [("train", train_df), ("test", test_df)]:
    df = (df
        .withColumn("Sched_month_sin",     sin(lit(2 * math.pi) * month(col("ScheduledDay"))      / 12))
        .withColumn("Sched_month_cos",     cos(lit(2 * math.pi) * month(col("ScheduledDay"))      / 12))
        .withColumn("Sched_dayofyear_sin", sin(lit(2 * math.pi) * dayofyear(col("ScheduledDay"))  / 365))
        .withColumn("Sched_dayofyear_cos", cos(lit(2 * math.pi) * dayofyear(col("ScheduledDay"))  / 365))
        .withColumn("Appoi_month_sin",     sin(lit(2 * math.pi) * month(col("AppointmentDay"))    / 12))
        .withColumn("Appoi_month_cos",     cos(lit(2 * math.pi) * month(col("AppointmentDay"))    / 12))
        .withColumn("Appoi_dayofyear_sin", sin(lit(2 * math.pi) * dayofyear(col("AppointmentDay"))/ 365))
        .withColumn("Appoi_dayofyear_cos", cos(lit(2 * math.pi) * dayofyear(col("AppointmentDay"))/ 365))
        .drop("ScheduledDay", "AppointmentDay")   # raw date columns not usable by VectorAssembler
    )

    if df_name == "train":
        train_df = df
    else:
        test_df = df

# Add the cyclical columns to the feature list defined in the cell above
numerical_cols += [
    "Sched_month_sin", "Sched_month_cos",
    "Sched_dayofyear_sin", "Sched_dayofyear_cos",
    "Appoi_month_sin", "Appoi_month_cos",
    "Appoi_dayofyear_sin", "Appoi_dayofyear_cos",
]

# silver_df.select(*numerical_cols[-8:]).display()

In [0]:
%skip
# Clean up old model references to free cache space
import gc
try:
    del scalerModel, model1, model2
except NameError:
    pass
gc.collect()


### Encode Neighborhood

In [0]:
# Encode neighborhood before scaling

from sklearn.preprocessing import TargetEncoder
from pyspark.sql.functions import col, create_map, lit
from itertools import chain
import pandas as pd

# display(train_df.limit(5))

# 1. Collect training data to pandas — fit encoder on train only
train_pd = train_df.select("Neighborhood", "Showed_up").toPandas()

enc = TargetEncoder(target_type="binary", smooth="auto")
enc.fit(train_pd[["Neighborhood"]], train_pd["Showed_up"])

# 2. Build a lookup map: Neighborhood string - encoded float
categories   = enc.categories_[0]                  # array of Neighborhood names
encoded_vals = enc.transform(pd.DataFrame({"Neighborhood": categories}))

mapping = dict(zip(categories, encoded_vals[:, 0].tolist()))

# 3. Apply the map to both splits as a new Spark column
map_expr = create_map([lit(x) for x in chain.from_iterable(mapping.items())])

train_df = train_df.withColumn("Neighborhood_te", map_expr[col("Neighborhood")])
test_df  = test_df.withColumn("Neighborhood_te", map_expr[col("Neighborhood")])

# 4. Add to your numerical features and drop the raw string column
numerical_cols += ["Neighborhood_te"]
train_df = train_df.drop("Neighborhood")
test_df  = test_df.drop("Neighborhood")

%md
### Encode Gender

In [0]:
from pyspark.ml.feature import OneHotEncoder, StringIndexer, VectorAssembler, MinMaxScaler
from pyspark.ml import Pipeline

# Encode only the 'Gender' column
indexer = StringIndexer(inputCol="Gender", outputCol="Gender_index", handleInvalid="skip")
encoder = OneHotEncoder(inputCol="Gender_index", outputCol="Gender_vec")

numerical_cols += ["Gender_vec"]

### Bring Gender vector into training and test data

### Transform the Dataframes

In [0]:
# Add Gender_vec to both dataframes

# Fit indexer on train, transform both train and test
indexer_model = indexer.fit(train_df)
train_df = indexer_model.transform(train_df)
test_df = indexer_model.transform(test_df)

# Fit encoder on train, transform both train and test
encoder_model = encoder.fit(train_df)
train_df = encoder_model.transform(train_df)
test_df = encoder_model.transform(test_df)

### Assemble feature vector with VectorAssembler

In [0]:
# Assemble features for both splits
vector_assembler = VectorAssembler(
    inputCols=numerical_cols,
    outputCol="features",
    handleInvalid="skip",
)


train_assembled = vector_assembler.transform(train_df)
test_assembled = vector_assembler.transform(test_df)


#### Non-date numeric values

In [0]:
# Fit scaler on TRAIN only to prevent leakage
scaler = MinMaxScaler(inputCol="features", outputCol="scaledFeatures")
scalerModel = scaler.fit(train_assembled)


# Transform both splits
train_scaled = scalerModel.transform(train_assembled)
test_scaled = scalerModel.transform(test_assembled)

#### Handle class imbalance
Create a weight column, apply heavy weight to minority class and low weight to majority class

In [0]:
# Find count of no-show appointments and count of show appointments. find quotient and make weights.
from pyspark.sql.functions import col

no_show_count = train_scaled.filter(col("Showed_up") == 0).count()
show_count = train_scaled.filter(col("Showed_up") == 1).count()
total_count = train_scaled.count()

WEIGHT_SHOWED_UP = 1.0
WEIGHT_NO_SHOW = show_count / no_show_count

In [0]:
from pyspark.sql.functions import when

train_scaled = train_scaled.withColumn(
    "weightCol",
    when(col("Showed_up") == 1, WEIGHT_SHOWED_UP).
    otherwise(WEIGHT_NO_SHOW)
)

In [0]:
print(f"Features assembled and scaled")
print(f"  Train: {train_scaled.count():,} rows, {len(numerical_cols)} features")
print(f"  Test: {test_scaled.count():,} rows")

#### Hyperparameter tuning

#### Apply hyperparameter grid
ParamGridBuilder: https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.tuning.ParamGridBuilder.html

In [0]:
import numpy as np
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator, CrossValidatorModel

In [0]:
# Scorer
from pyspark.ml.evaluation import BinaryClassificationEvaluator

evaluator = BinaryClassificationEvaluator(
    labelCol = "Showed_up",
    metricName = "areaUnderROC"
)


### CrossValidator Documentation
https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.tuning.CrossValidator.html

In [0]:
import os
os.environ["SPARKML_TEMP_DFS_PATH"] = "/Volumes/workspace/default/no_show_volume"

In [0]:
%skip
from sklearn.model_selection import GridSearchCV
clf = GridSearchCV(lr, param_grid = rf_model, lr_cv=3, verbose = True, n_jobs = 1)
clf

In [0]:
%skip
# best_clf = clf.fit(x,y) -- featuresCol, labelCol
best_clf = clf.fit(features, Showed_up)
best_clf = clf.best_estimator_

### Random Forest Classifier

In [0]:
from pyspark.ml.classification import RandomForestClassifier

rf = RandomForestClassifier(featuresCol = 'scaledFeatures',
                            labelCol = 'Showed_up',
                            predictionCol = 'prediction',
                            weightCol = 'weightCol'
)

#### Create RF Param grid

In [0]:
%skip
rf_param_grid = (ParamGridBuilder() \
    .addGrid(rf.numTrees, [50, 100, 200]) \
    .addGrid(rf.maxDepth, [5, 10, 15]) \
    .addGrid(rf.minInstancesPerNode, [1, 10]) \
    .build()
) # End param_grid

In [0]:
rf_param_grid = (ParamGridBuilder() \
    .addGrid(rf.numTrees, [50, 100]) \
    .addGrid(rf.maxDepth, [5, 10]) \
    .build()
) # End param_grid

#### RF CrossValidator

In [0]:
cv_rf = CrossValidator(
    estimator = rf,
    estimatorParamMaps = rf_param_grid,
    evaluator = evaluator,
    numFolds = 3,
    parallelism = 2
)

In [0]:
# pipeline = Pipeline(stages=[indexer, encoder, vector_assembler])

stages2=[indexer, encoder, vector_assembler, rf]
pipeline2 = Pipeline().setStages(stages2)

In [0]:
train_df = spark.read.table('train_df_tbl')
test_df = spark.read.table('test_df_tbl')

In [0]:
%skip
# train_scaled already has indexing/encoding applied, fit LR directly
from pyspark.ml.classification import LogisticRegression

lr_model = lr.fit(train_scaled)
lr_predictions = lr_model.transform(test_scaled)

In [0]:
%skip
# # Free model cache from previous LR CrossValidator run
# for _v in ("cvModel", "lr_model", "lr_predictions"):
#     if _v in globals():
#         del globals()[_v]
# gc.collect()

In [0]:
import gc

# train_scaled already has indexing/encoding applied, fit RF directly
rf_model = cv_rf.fit(train_scaled)
rf_predictions = rf_model.transform(test_scaled)

In [0]:
%skip
help(pipeline)

#### Logistic Regression Baseline

No hyperparameter tuning - fast, interpretable baseline to anchor subsequent model comparison

In [0]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.sql import functions as F

In [0]:
import mlflow       # Experiment tracking & Record ML runs
import mlflow.spark # Log model artifacts
import os
from mlflow.models import infer_signature

os.environ["MLFLOW_DFS_TMP"] = "/Volumes/workspace/default/no_show_volume"

# EXPERIMENT_NAME = "/Users/asanders4205@gmail.com/predict_no_show"
# TARGET = "Showed_up"

In [0]:
#  columns: prediction, label, weight (optional) and probabilityCol (only for logLoss)
multi_evaluator = MulticlassClassificationEvaluator(
    predictionCol = 'prediction',
    labelCol = 'Showed_up'
)

In [0]:
signature = infer_signature(train_df, rf_predictions)

### Evaluate and log RF metrics

In [0]:
# help(RandomForestClassifier)
# print(multi_evaluator.explainParams())
precision = multi_evaluator.evaluate(rf_predictions, {multi_evaluator.metricName: "weightedPrecision"})
recall = multi_evaluator.evaluate(rf_predictions, {multi_evaluator.metricName: "weightedRecall"})
f1 = multi_evaluator.evaluate(rf_predictions, {multi_evaluator.metricName: "f1"})
# auc = multi_evaluator.evaluate(rf_predictions, {multi_evaluator.metricName: "AreaUnderAUC"})
# accuracy = multi_evaluator.evaluate(rf_predictions, {multi_evaluator.metricName: "accuracy"})
# support = multi_evaluator.evaluate(rf_predictions, {multi_evaluator.metricName: "support"})

In [0]:
# Use the signature already inferred in cell 70 (train_df → rf_predictions)
# input_example would fail here due to non-JSON-serializable Vector columns

with mlflow.start_run(run_name="RandomForestModel") as run:
    # Log model artifact with pre-computed signature
    mlflow.spark.log_model(rf_model, 
                           "random_forest_model",
                           signature=signature  # From cell 70
    ) # End log model
    mlflow.log_param("pip_requirements", ["pyspark==4.1.0"])
    mlflow.log_param("maxIter", 10)
    mlflow.log_param("featuresCol", "features")
    mlflow.log_param("labelCol", "Showed_up")
    mlflow.log_metric("f1", f1)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)

### Log Hyperparameters